# CloudFormation & IaC

## CloudFormation Basics

**CloudFormation** is AWS's IaC service. You write a template describing resources (EC2, S3, RDS, etc.), and CloudFormation creates and manages them as a **stack**.

Templates are JSON or YAML files. CloudFormation handles dependencies, rollback on failure, and updates.

## Template Structure

A CloudFormation template has these sections:

**AWSTemplateFormatVersion** specifies the template version (usually "2010-09-09").

**Description** explains the template.

**Parameters** are input values you provide when creating the stack.

**Resources** define the AWS resources to create.

**Outputs** return values from the stack (e.g., instance IP, database endpoint).

**Mappings** define lookup tables for conditional values.

## Stack Operations

**Create** a stack from a template. CloudFormation provisions all resources.

**Update** a stack to change resources. CloudFormation applies changes with minimal downtime.

**Delete** a stack to remove all resources (with some exceptions like S3 buckets with data).

**Drift detection** identifies resources that were manually changed outside CloudFormation.

## Hands-On: Create CloudFormation Stack

Create a template file (template.yaml):

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'Simple EC2 instance'

Parameters:
  InstanceType:
    Type: String
    Default: t3.micro
    Description: EC2 instance type

Resources:
  MyInstance:
    Type: AWS::EC2::Instance
    Properties:
      ImageId: ami-0c55b159cbfafe1f0
      InstanceType: !Ref InstanceType
      Tags:
        - Key: Name
          Value: MyInstance

Outputs:
  InstanceId:
    Value: !Ref MyInstance
    Description: Instance ID
  InstancePublicIp:
    Value: !GetAtt MyInstance.PublicIp
    Description: Public IP address
```

Create the stack:

```bash
aws cloudformation create-stack --stack-name my-stack \
  --template-body file://template.yaml \
  --parameters ParameterKey=InstanceType,ParameterValue=t3.small
```

Describe stack:

```bash
aws cloudformation describe-stacks --stack-name my-stack
```

Get stack outputs:

```bash
aws cloudformation describe-stacks --stack-name my-stack \
  --query 'Stacks[0].Outputs'
```

Detect drift:

```bash
aws cloudformation detect-stack-drift --stack-name my-stack
aws cloudformation describe-stack-drift-detection-status \
  --stack-drift-detection-id drift-id
```

## Python Boto3 Example

In [ ]:
import boto3
import json

cf = boto3.client('cloudformation')

# Create stack
cf.create_stack(
    StackName='my-stack',
    TemplateBody=open('template.yaml').read(),
    Parameters=[
        {'ParameterKey': 'InstanceType', 'ParameterValue': 't3.micro'}
    ]
)

# Describe stack
response = cf.describe_stacks(StackName='my-stack')
for stack in response['Stacks']:
    print(f"Status: {stack['StackStatus']}")
    for output in stack.get('Outputs', []):
        print(f"{output['OutputKey']}: {output['OutputValue']}")

# Update stack
cf.update_stack(
    StackName='my-stack',
    TemplateBody=open('template.yaml').read()
)

# Delete stack
cf.delete_stack(StackName='my-stack')

## Advanced Template Example

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'VPC with public and private subnets'

Parameters:
  VpcCidr:
    Type: String
    Default: 10.0.0.0/16

Resources:
  VPC:
    Type: AWS::EC2::VPC
    Properties:
      CidrBlock: !Ref VpcCidr

  PublicSubnet:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.1.0/24
      AvailabilityZone: us-east-1a

  InternetGateway:
    Type: AWS::EC2::InternetGateway

  AttachGateway:
    Type: AWS::EC2::VPCGatewayAttachment
    Properties:
      VpcId: !Ref VPC
      InternetGatewayId: !Ref InternetGateway

  PublicRouteTable:
    Type: AWS::EC2::RouteTable
    Properties:
      VpcId: !Ref VPC

  PublicRoute:
    Type: AWS::EC2::Route
    Properties:
      RouteTableId: !Ref PublicRouteTable
      DestinationCidrBlock: 0.0.0.0/0
      GatewayId: !Ref InternetGateway

Outputs:
  VpcId:
    Value: !Ref VPC
  SubnetId:
    Value: !Ref PublicSubnet
```

## Terraform Example

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = var.instance_type

  tags = {
    Name = "MyInstance"
  }
}

variable "instance_type" {
  type    = string
  default = "t3.micro"
}

output "instance_id" {
  value = aws_instance.web.id
}

output "public_ip" {
  value = aws_instance.web.public_ip
}
```

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is CloudFormation?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>AWS's Infrastructure as Code service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>A database service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>A compute service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>A networking service</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is a CloudFormation stack?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>A template file</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>A collection of AWS resources created from a template</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>A parameter in a template</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>A backup of resources</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is drift detection?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>A template validation feature</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>A backup mechanism</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>Identifying resources manually changed outside CloudFormation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>A cost optimization feature</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What are CloudFormation parameters?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="0">
      <span>Input values provided when creating a stack</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="1">
      <span>Output values from a stack</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="2">
      <span>AWS resources</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="3">
      <span>Template metadata</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What are CloudFormation outputs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="0">
      <span>Input parameters</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="1">
      <span>Return values from a stack (e.g., instance IP, database endpoint)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="2">
      <span>Error messages</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="3">
      <span>Resource tags</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>